## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-04-03-54-05 +0000,nyt,Annie Andrews Defeats Darline Graham in South ...,https://www.nytimes.com/2026/11/03/us/andrews-...
1,2026-11-04-03-52-50 +0000,nyt,Acton Wins Governor’s Race in Ohio,https://www.nytimes.com/2026/11/03/us/ohio-gov...
2,2026-11-04-03-51-28 +0000,missionlocal,San Francisco 2026 election results: Live updates,https://missionlocal.org/2026/11/san-francisco...
3,2026-11-04-03-50-55 +0000,chitri,Bill Foster wins re-election in Illinois 11th ...,https://www.chicagotribune.com/2026/11/03/illi...
4,2026-11-04-03-50-32 +0000,chitri,Kevin Hagen leads in race for Center Township ...,https://www.chicagotribune.com/2026/11/03/kevi...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2714/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
15,election,47
7,senate,33
32,trump,31
102,democrats,26
8,race,25


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
166,2026-11-03-22-49-00 +0000,wsj,Republicans and Democrats face off in a high-s...,https://www.wsj.com/politics/policy/democrats-...,218
37,2026-11-04-03-14-56 +0000,chitri,South Carolina Senate race: Democrat Annie And...,https://www.chicagotribune.com/2026/11/03/sout...,199
113,2026-11-04-00-55-30 +0000,chitri,North Carolina Senate race: Democrat Roy Coope...,https://www.chicagotribune.com/2026/11/03/nort...,182
26,2026-11-04-03-32-02 +0000,startribune,Democrat Roy Cooper wins North Carolina Senate...,https://www.startribune.com/roy-cooper-looks-t...,178
100,2026-11-04-01-04-38 +0000,chitri,Georgia Senate race: Sen. Jon Ossoff wins reel...,https://www.chicagotribune.com/2026/11/03/geor...,176


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
166,218,2026-11-03-22-49-00 +0000,wsj,Republicans and Democrats face off in a high-s...,https://www.wsj.com/politics/policy/democrats-...
37,199,2026-11-04-03-14-56 +0000,chitri,South Carolina Senate race: Democrat Annie And...,https://www.chicagotribune.com/2026/11/03/sout...
154,105,2026-11-03-23-18-22 +0000,nypost,Midterms 2026 elections results map: Live upda...,https://nypost.com/2026/11/03/us-news/midterms...
13,98,2026-11-04-03-42-02 +0000,startribune,"Georgia Sen. Jon Ossoff wins reelection, defea...",https://www.startribune.com/sen-jon-ossoff-tri...
92,82,2026-11-04-01-21-31 +0000,nypost,Chris Pappas halts Sununu comeback in New Hamp...,https://nypost.com/2026/11/03/us-news/chris-pa...
350,64,2026-11-03-13-34-20 +0000,chitri,Photos: Voters head to the polls on Election D...,https://www.chicagotribune.com/2026/11/03/phot...
399,54,2026-11-03-10-56-00 +0000,wsj,Trump surprises Republicans with an election-e...,https://www.wsj.com/politics/elections/control...
145,51,2026-11-03-23-39-55 +0000,nyt,OpenAI and Anthropic Set Up ‘War Rooms’ to Fig...,https://www.nytimes.com/2026/11/03/business/ai...
113,49,2026-11-04-00-55-30 +0000,chitri,North Carolina Senate race: Democrat Roy Coope...,https://www.chicagotribune.com/2026/11/03/nort...
76,48,2026-11-04-02-06-00 +0000,wsj,Democrats jumped out to a strong start Tuesday...,https://www.wsj.com/politics/policy/democrats-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
